In [93]:
def PWF(N, i, d):
    if i != d:
        return 1 / (d - i) * (1 - ((1 + i) / (1 + d)) ** N)
    elif i == d:
        return N / (1 + i)


def P1(N, i, d, c, t):
    P1 = (1 - c * t) * PWF(N, i, d)
    return P1


def C1(annual_load, electric_price):
    C1 = annual_load * electric_price
    return C1


def P2(DP, R, N_1, d, N_L, loan_interest, M, c, t, N, M_i):
    "DP=downpayment ratio, R=tax rebate,N_1 = min(loan length, analysis length),c=taxdeductible T/F, t=tax rate,N=analysis years, M_i=maintenance interest rate,d=market discount rate"
    P2 = DP - R + (1 - DP) * PWF(N_1, 0, d) / PWF(N_L, 0, loan_interest) + M * (1 - c * t) * PWF(N, M_i, d)

    print((1 - DP) * PWF(N_1, 0, d) / PWF(N_L, 0, loan_interest))
    print(M * (1 - c * t) * PWF(N, M_i, d))
    return P2

In [94]:
# electric_price = 0.25  # $/kWh
electric_price_MWH = 35.90  #$/MWh
electric_i = 0.015     # Electric inflation
wind_farm = 6.9e9*0    #cost to build
solar_farm = 7.81e9*2   #cost
wind_energy = 9.8e12
solar_energy = 12.3e12
total_energy = wind_energy+solar_energy #in watts
total_energy_MWH = total_energy/(10e6)
C2 = wind_farm + solar_farm
# farm_cost = wind_C2 = 45e6
N_L = 20  # year, loan length

loan_i = 0.05
DP = 0.25   # Downpayment ratio
M = 0.04    # Mainteance ration
d = 0.07    # Discount rate
t = c = 0  # tax rate and T/F deduction
R= 0.3     # Rebate
R=0.3
print(C2)


15620000000.0


In [114]:
def find_LCS(N, verbose=False, farm_C2=C2):
    N_1 = min(N, N_L)
    farm_P2 = P2(DP, R, N_1, d, N_L, loan_i, M, c, t, N, M_i=electric_i)  # What is the maintenance interest
    # print(f'Farm Energy Cost: ${wind_P1*wind_C1}')
    farm_LCC = farm_P2 * farm_C2
    if verbose:
        print(f"Farm P2: {farm_P2:,.2F}")
        print(f"Farm C2: {farm_C2:,.2F}")
        print(f"Farm LCC: ${farm_LCC:,.2F}")
    
    grid_P1 = P1(N, electric_i, d, c, t)
    # grid_C1 = annual_load*1000*electric_price
    grid_C1 = total_energy_MWH  * electric_price_MWH
    grid_LCC = grid_P1 * grid_C1
    if verbose:
        print(f"Grid C1: ${grid_C1:,.2F}")
        print(f"Grid P1: {grid_P1:,.2F}")
        print(f"Grid Cost: ${grid_LCC:,.2F}")
        print(f"Total energy rated: ${total_energy_MWH:,.2F} MWH")

    LCS = grid_LCC - farm_LCC
    wind_LCOE = farm_LCC / (total_energy_MWH * N)
    if verbose:
        print(f"Savings: ${round(LCS,2):,.2F}")
        print(f"LCOE: ${round(wind_LCOE,5):,.2F}")
    return LCS, wind_LCOE


N = 25  # year, analysis lenght

LCS, wind_LCOE = find_LCS(N=25, verbose=True)

0.637568333846442
0.5328469177279235
Farm P2: 1.12
Farm C2: 15,620,000,000.00
Farm LCC: $17,500,886,229.59
Grid C1: $79,339,000.00
Grid P1: 13.32
Grid Cost: $1,056,888,540.14
Total energy rated: $2,210,000.00 MWH
Savings: $-16,443,997,689.45
LCOE: $316.76


In [83]:
from scipy.optimize import fsolve

def payback_period(N):
    LCS, wind_LCOE = find_LCS(N=N, verbose=False)
    return LCS

fsolve(payback_period,x0=[21])

array([0.55487941])

In [115]:
from scipy.optimize import fsolve

def positive_savings_where(farm_cost):
    LCS, wind_LCOE = find_LCS(N=N, verbose=False,farm_C2=farm_cost)
    return LCS

fsolve(positive_savings_where,x0=[21])

0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235
0.637568333846442
0.5328469177279235


array([9.43300744e+08])

In [124]:
find_LCS(N=30, verbose=True,farm_C2=9.430e+08)

0.637568333846442
0.5779364782201867
Farm P2: 1.17
Farm C2: 943,000,000.00
Farm LCC: $1,099,071,037.78
Grid C1: $79,339,000.00
Grid P1: 14.45
Grid Cost: $1,146,322,556.14
Total energy rated: $2,210,000.00 MWH
Savings: $47,251,518.36
LCOE: $16.58


(47251518.35895395, 16.577240388820975)

In [123]:
def payback_period(N):
    LCS, wind_LCOE = find_LCS(N=N, verbose=False,farm_C2=9.430e+08)
    return LCS

fsolve(payback_period,x0=[21])

0.637568333846442
[0.48715465]
0.637568333846442
[0.48715465]
0.637568333846442
[0.48715465]
0.637568333846442
[0.48715465]
0.637568333846442
[0.52849472]
0.637568333846442
[0.532121]
0.637568333846442
[0.53251897]
0.637568333846442
[0.53252306]
0.637568333846442
[0.53252307]
0.637568333846442
[0.53252307]


array([24.96846148])